# 🏠 Ames Housing データで学ぶ本格的な回帰分析
## ― Raschka (2022) *Machine Learning with PyTorch and Scikit-Learn* ch09 拡張版 ―

**データ出典：** Dean De Cock (2011). *Ames, Iowa: Alternative to the Boston Housing Data as an End of Year Statistics Project.*  
→ Raschka のリポジトリ `rasbt/machine-learning-book` から直接取得

| 章 | 内容 |
|----|------|
| 1 | データ取得・前処理（欠損・エンコーディング・対数変換） |
| 2 | EDA：散布図行列・相関ヒートマップ |
| 3 | OLS 重回帰（statsmodels）・係数解釈 |
| 4 | **多重共線性**の検出と対処（VIF・条件数） |
| 5 | **回帰診断**：残差・等分散・正規性・外れ値・影響点 |
| 6 | **不均一分散**の検定と対処（BP 検定・WLS・対数変換） |
| 7 | **Ridge / Lasso / Elastic Net** 正則化 |
| 8 | **AIC / BIC** によるモデル選択・ステップワイズ法 |
| 9 | 交差検証・最終モデル評価 |

---

## 0. ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# statsmodels ── 本格的な統計的回帰分析
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.gofplots import ProbPlot
from statsmodels.stats.outliers_influence import summary_table

# scikit-learn ── 正則化・CV
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline

import warnings
warnings.filterwarnings('ignore')

import matplotlib
for font in ['IPAexGothic', 'Noto Sans CJK JP', 'Hiragino Sans']:
    try: matplotlib.rc('font', family=font); break
    except: pass
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

print('準備完了 ✅')
print(f'statsmodels: {sm.__version__}')


---
## 1. データ取得と前処理

### データ概要
Ames Housing データセット（2930 件、82 変数）はアイオワ州エームズ市の住宅売買記録。  
Raschka ch09 と同じ URL からタブ区切りで読み込む。

**目的変数：** `SalePrice`（住宅売買価格、単位：ドル）

| 説明変数 | 意味 | 多重共線性のリスク |
|---------|------|------------------|
| `Overall Qual` | 総合品質（1-10） | ★★ |
| `Overall Cond` | 総合状態（1-10） | ★ |
| `Gr Liv Area` | 地上居住面積（SF） | **★★★**（TotRms と高相関）|
| `Total Bsmt SF` | 地下室面積（SF） | **★★★**（1st Flr SF と高相関）|
| `1st Flr SF` | 1階面積（SF） | **★★★**（Total Bsmt SF と高相関）|
| `Garage Area` | ガレージ面積（SF） | **★★★**（Garage Cars と高相関）|
| `Garage Cars` | ガレージ台数 | **★★★**（Garage Area と高相関）|
| `Full Bath` | フルバス数 | ★★ |
| `TotRms AbvGrd` | 総室数（地上） | **★★★**（Gr Liv Area と高相関）|
| `Year Built` | 建築年 | ★★ |
| `Lot Area` | 敷地面積（SF） | ★ |
| `Central Air` | 中央空調ダミー | ★ |


In [ ]:
# ── Raschka と同じ URL から取得 ──────────────────────────────────
URL = 'https://raw.githubusercontent.com/rasbt/machine-learning-book/main/ch09/AmesHousing.txt'
df_raw = pd.read_csv(URL, sep='\t')
print(f'元データ: {df_raw.shape[0]:,} 行 × {df_raw.shape[1]} 列')

# ── 使用する変数を選択 ──────────────────────────────────────────
FEATURE_COLS = [
    'Overall Qual', 'Overall Cond', 'Gr Liv Area', 'Central Air',
    'Total Bsmt SF', '1st Flr SF', 'Garage Area', 'Garage Cars',
    'Full Bath', 'TotRms AbvGrd', 'Year Built', 'Lot Area'
]
TARGET = 'SalePrice'

df = df_raw[FEATURE_COLS + [TARGET]].copy()

# カテゴリ変数のエンコーディング
df['Central Air'] = df['Central Air'].map({'N': 0, 'Y': 1})

# 欠損値処理
print(f'欠損行数（処理前）: {df.isnull().any(axis=1).sum()}')
df = df.dropna()
print(f'分析用データ: {df.shape[0]:,} 行 × {df.shape[1]} 列')
df.describe().round(0)


In [ ]:
# ── SalePrice の分布確認 ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df[TARGET], bins=50, color='#2980b9', edgecolor='white', alpha=0.85)
axes[0].set(xlabel='SalePrice (USD)', ylabel='件数', title='原系列の分布')

axes[1].hist(np.log(df[TARGET]), bins=50, color='#27ae60', edgecolor='white', alpha=0.85)
axes[1].set(xlabel='log(SalePrice)', title='対数変換後の分布')

sm.qqplot(np.log(df[TARGET]), line='s', ax=axes[2], alpha=0.5)
axes[2].set_title('Q-Q プロット（log変換後）')

plt.suptitle('目的変数 SalePrice の分布', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'歪度（原系列）: {df[TARGET].skew():.3f}')
print(f'歪度（対数変換）: {np.log(df[TARGET]).skew():.3f}')


In [ ]:
# 対数変換した目的変数を追加
df['log_SalePrice'] = np.log(df[TARGET])


---
## 2. 探索的データ分析（EDA）

In [ ]:
# 散布図行列（主要変数に絞る）
plot_cols = ['log_SalePrice', 'Overall Qual', 'Gr Liv Area',
             'Total Bsmt SF', '1st Flr SF', 'Garage Area', 'Garage Cars']

g = sns.pairplot(df[plot_cols], diag_kind='kde', plot_kws={'alpha':0.3, 's':10})
g.fig.suptitle('散布図行列（主要変数）', y=1.01, fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# 相関行列ヒートマップ
corr = df[FEATURE_COLS + ['log_SalePrice']].corr().round(2)

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('相関行列ヒートマップ（log_SalePrice を含む）', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# 高相関ペアの一覧
print('💡 |r| > 0.70 の変数ペア（多重共線性の候補）:')
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i,j]
        if abs(r) > 0.70 and corr.columns[j] != 'log_SalePrice':
            print(f'  {corr.columns[i]:22s} ↔ {corr.columns[j]:22s}  r = {r:.2f}')


---
## 3. OLS 重回帰（statsmodels）

$$y_i = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_p x_{ip} + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0,\sigma^2)$$

目的変数は **log(SalePrice)** を使用（分布の正規化のため）。


In [ ]:
y      = df['log_SalePrice']
X      = df[FEATURE_COLS]
X_sm   = sm.add_constant(X)

ols = sm.OLS(y, X_sm).fit()
print(ols.summary())


In [ ]:
# 係数の信頼区間プロット（定数項を除く）
fig, ax = plt.subplots(figsize=(9, 6))
coef    = ols.params[1:]
ci      = ols.conf_int().iloc[1:]
colors  = ['#c0392b' if lo > 0 or hi < 0 else '#7f8c8d'
           for lo, hi in zip(ci[0], ci[1])]  # 0を含まなければ有意

y_pos = np.arange(len(coef))
ax.barh(y_pos, coef.values, xerr=[(coef - ci[0]).values, (ci[1] - coef).values],
        color=colors, alpha=0.8, capsize=4, edgecolor='white')
ax.axvline(0, color='black', lw=1, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(coef.index, fontsize=10)
ax.set_xlabel('係数（95% 信頼区間）')
ax.set_title('OLS 係数プロット（赤=有意、灰=非有意）', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 4. 多重共線性の検出と対処

### 4-1. VIF（分散膨張因子）

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

| VIF | 判定 |
|-----|------|
| < 5 | 問題なし |
| 5〜10 | 要注意 |
| > 10 | 多重共線性あり |
| > 100 | 深刻 |

### 4-2. 条件数（Condition Number）

$$\kappa = \sqrt{\frac{\lambda_{\max}}{\lambda_{\min}}}$$

$X^\top X$ の最大・最小固有値の比の平方根。**30 以上**で多重共線性が疑われる。


In [ ]:
# VIF の計算と可視化
vif_vals = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_df   = pd.DataFrame({'変数': FEATURE_COLS, 'VIF': vif_vals}).sort_values('VIF', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#c0392b' if v > 10 else '#e67e22' if v > 5 else '#27ae60' for v in vif_df['VIF']]
bars   = ax.barh(vif_df['変数'], vif_df['VIF'], color=colors, edgecolor='white')
for bar, val in zip(bars, vif_df['VIF']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
ax.axvline(5,  color='orange', linestyle='--', lw=1, label='VIF=5')
ax.axvline(10, color='red',    linestyle='--', lw=1.5, label='VIF=10')
ax.set_xlabel('VIF')
ax.set_title('VIF（赤=多重共線性あり）', fontsize=12, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

# 条件数
cond_num = np.linalg.cond(X_sm.values)
print(f'\n条件数 κ = {cond_num:.1f}  （30以上で多重共線性を疑う）')
print(vif_df.to_string(index=False))


In [ ]:
# ── 多重共線性の原因変数を除去したモデル ──────────────────────────
# '1st Flr SF'（Total Bsmt SF と高相関）と 'Garage Cars'（Garage Area と高相関）を除く
FEAT_REDUCED = [c for c in FEATURE_COLS if c not in ('1st Flr SF', 'Garage Cars', 'TotRms AbvGrd')]
X_red = sm.add_constant(df[FEAT_REDUCED])

ols_red = sm.OLS(y, X_red).fit()

# VIF 再計算
vif_red = [variance_inflation_factor(df[FEAT_REDUCED].values, i)
           for i in range(len(FEAT_REDUCED))]
vif_red_df = pd.DataFrame({'変数': FEAT_REDUCED, 'VIF': vif_red}).sort_values('VIF', ascending=False)
print('変数削減後の VIF:')
print(vif_red_df.to_string(index=False))
print(f'\n条件数（削減後）: {np.linalg.cond(X_red.values):.1f}')
print(f'AIC: {ols.aic:.1f} → {ols_red.aic:.1f}  (full → reduced)')


---
## 5. 回帰診断

OLS の4大仮定（LINE）を体系的に検証する。

| 仮定 | 内容 | 検定・確認方法 |
|------|------|--------------|
| **L**inearity | 線形性 | RESET 検定・残差 vs 予測値 |
| **I**ndependence | 独立性 | Durbin-Watson 検定 |
| **N**ormality | 残差の正規性 | Q-Q プロット・Jarque-Bera 検定 |
| **E**qual Variance | 等分散性（均一分散） | Breusch-Pagan 検定・White 検定 |


In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)

resid   = ols_red.resid
fitted  = ols_red.fittedvalues
stdresid = ols_red.get_influence().resid_studentized_internal

# ① 残差 vs 予測値
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(fitted, resid, alpha=0.3, s=10, color='#2980b9')
ax1.axhline(0, color='red', lw=1.2, linestyle='--')
ax1.set(xlabel='予測値 log(SalePrice)', ylabel='残差', title='① 残差 vs 予測値')

# ② Q-Q プロット
ax2 = fig.add_subplot(gs[0, 1])
pp  = ProbPlot(resid)
pp.qqplot(line='s', ax=ax2, alpha=0.3, markersize=3, color='#2980b9')
ax2.set_title('② Q-Q プロット（残差）')

# ③ Scale-Location（√|標準化残差| vs 予測値）
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(fitted, np.sqrt(np.abs(stdresid)), alpha=0.3, s=10, color='#27ae60')
ax3.set(xlabel='予測値', ylabel='√|標準化残差|', title='③ Scale-Location')

# ④ 残差のヒストグラム
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(resid, bins=50, color='#2980b9', edgecolor='white', alpha=0.85)
ax4.set(xlabel='残差', ylabel='件数', title='④ 残差のヒストグラム')

# ⑤ Cook's D（影響点）
influence = OLSInfluence(ols_red)
cooks_d   = influence.cooks_distance[0]
ax5 = fig.add_subplot(gs[1, 1])
ax5.stem(np.arange(len(cooks_d)), cooks_d, markerfmt=',', linefmt='grey', basefmt='k-')
threshold = 4 / len(df)
ax5.axhline(threshold, color='red', lw=1.5, linestyle='--', label=f'4/n = {threshold:.4f}')
ax5.set(xlabel='観測番号', ylabel="Cook's D", title="⑤ Cook's Distance")
ax5.legend(fontsize=9)

# ⑥ Leverage vs 標準化残差（影響点の可視化）
leverage = influence.hat_matrix_diag
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(leverage, stdresid, alpha=0.4, s=12, color='#9b59b6')
ax6.axhline(0, color='black', lw=0.8, linestyle='--')
ax6.axhline(2,  color='orange', lw=1, linestyle='--')
ax6.axhline(-2, color='orange', lw=1, linestyle='--')
ax6.set(xlabel='Leverage', ylabel='標準化残差', title='⑥ Leverage vs 標準化残差')

plt.suptitle('回帰診断プロット（6種類）', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── 各診断の数値検定 ──────────────────────────────────────────
print('=' * 55)
print('  回帰診断 数値検定サマリ')
print('=' * 55)

# ① 線形性: RESET 検定
reset  = linear_reset(ols_red, power=3, use_f=True)
print(f'\n① RESET 検定（線形性）')
print(f'   F統計量 = {reset.statistic:.3f},  p値 = {reset.pvalue:.4f}')
print(f'   → {"線形性に問題あり (p<0.05)" if reset.pvalue < 0.05 else "線形性に問題なし"}')

# ② 自己相関: Durbin-Watson
dw = durbin_watson(resid)
print(f'\n② Durbin-Watson 統計量 = {dw:.4f}')
print(f'   → 2に近いほど自己相関なし（1.5〜2.5が許容範囲）')

# ③ 正規性: Jarque-Bera（summary() 内に含まれる）
jb_stat, jb_p = ols_red.summary().tables[2].data[3][2], ols_red.summary().tables[2].data[3][3]
print(f'\n③ Jarque-Bera 検定（正規性）')
print(f'   JB統計量 = {ols_red.diagn["jb"]:,.2f},  p値 = {ols_red.diagn["jbpv"]:.6f}')
print(f'   → {"正規性に問題あり (p<0.05)" if ols_red.diagn["jbpv"] < 0.05 else "正規性に問題なし"}')

# ④ 等分散性: Breusch-Pagan + White
bp_lm, bp_p, _, _ = het_breuschpagan(resid, X_red)
print(f'\n④ Breusch-Pagan 検定（等分散性）')
print(f'   LM統計量 = {bp_lm:.3f},  p値 = {bp_p:.6f}')
print(f'   → {"不均一分散あり (p<0.05) → WLS or 対数変換を検討" if bp_p < 0.05 else "均一分散"}')

wh_lm, wh_p, _, _ = het_white(resid, X_red)
print(f'\n   White 検定（等分散性）')
print(f'   LM統計量 = {wh_lm:.3f},  p値 = {wh_p:.6f}')

# ⑤ 影響点の数
n_high_cook  = (cooks_d > 4/len(df)).sum()
n_high_lev   = (leverage > 2 * X_red.shape[1] / len(df)).sum()
print(f'\n⑤ 影響点')
print(f'   Cook\'s D > 4/n: {n_high_cook} 件')
print(f'   高 Leverage  : {n_high_lev} 件')


---
## 6. 不均一分散への対処

Breusch-Pagan 検定で不均一分散が検出された場合、3つのアプローチがある。

| アプローチ | 方法 | 特徴 |
|-----------|------|------|
| **対数変換** | 目的変数を対数化 | すでに実施済み。それでも残る場合は… |
| **WLS** | 重み付き最小二乗法 | 予測値の逆数などを重みに使う |
| **HC標準誤差** | ロバスト標準誤差 | 係数は OLS と同じ、SE のみ修正 |


In [ ]:
# ── HC3 ロバスト標準誤差（White の修正）────────────────────────
ols_hc3 = ols_red.get_robustcov_results(cov_type='HC3')
print('HC3 ロバスト標準誤差によるサマリ:')
print(ols_hc3.summary())


In [ ]:
# ── WLS（重み付き最小二乗法）─────────────────────────────────
# 第1段階：残差の分散を予測値の関数としてモデル化
resid_sq  = resid ** 2
aux_model = sm.OLS(np.log(resid_sq), X_red).fit()
fitted_var = np.exp(aux_model.fittedvalues)
weights   = 1.0 / fitted_var

wls = sm.WLS(y, X_red, weights=weights).fit()
print('WLS サマリ:')
print(wls.summary())

print(f'\nOLS R²  = {ols_red.rsquared:.4f}')
print(f'WLS R²  = {wls.rsquared:.4f}')


---
## 7. 正則化回帰

### 7-1. 正則化の理論

| 手法 | 目的関数 | 特徴 |
|------|---------|------|
| **Ridge** | $\|y-X\beta\|^2 + \lambda\|\beta\|^2$ | 係数を縮小（ゼロにはしない） |
| **Lasso** | $\|y-X\beta\|^2 + \lambda\|\beta\|_1$ | スパース解（変数選択） |
| **Elastic Net** | 両方の混合 | $l_1$比で Ridge と Lasso を補間 |


In [ ]:
# 正規化のため StandardScaler を適用
X_train, X_test, y_train, y_test = train_test_split(
    df[FEAT_REDUCED], y, test_size=0.3, random_state=123)

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

# 正則化パスの計算
alphas = np.logspace(-4, 2, 400)
ridge_coefs = np.array([Ridge(alpha=a).fit(X_tr_sc, y_train).coef_ for a in alphas])
lasso_coefs = np.array([Lasso(alpha=a, max_iter=20000).fit(X_tr_sc, y_train).coef_ for a in alphas])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_l  = plt.cm.tab10(np.linspace(0, 1, len(FEAT_REDUCED)))
for i, (feat, c) in enumerate(zip(FEAT_REDUCED, colors_l)):
    axes[0].plot(alphas, ridge_coefs[:, i], label=feat, color=c, lw=1.5)
    axes[1].plot(alphas, lasso_coefs[:, i], label=feat, color=c, lw=1.5)
for ax, title in zip(axes, ['Ridge (L2)', 'Lasso (L1)']):
    ax.set_xscale('log')
    ax.axhline(0, color='black', lw=0.7, linestyle='--', alpha=0.4)
    ax.set(xlabel='λ（対数スケール）', ylabel='標準化係数', title=f'{title} 正則化パス')
    ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()


In [ ]:
# CV で最適 λ を選択
kf = KFold(n_splits=10, shuffle=True, random_state=42)

ridge_cv = RidgeCV(alphas=alphas, cv=kf).fit(X_tr_sc, y_train)
lasso_cv = LassoCV(alphas=alphas, cv=kf, max_iter=20000, random_state=42).fit(X_tr_sc, y_train)
enet_cv  = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95],
                        alphas=alphas, cv=kf, max_iter=20000, random_state=42).fit(X_tr_sc, y_train)

print(f'Ridge     最適 λ = {ridge_cv.alpha_:.5f}')
print(f'Lasso     最適 λ = {lasso_cv.alpha_:.5f}')
print(f'ElasticNet最適 λ = {enet_cv.alpha_:.5f},  l1_ratio = {enet_cv.l1_ratio_:.2f}')

lasso_zero = [FEAT_REDUCED[i] for i, c in enumerate(lasso_cv.coef_) if abs(c) < 1e-8]
print(f'\nLasso がゼロに落とした変数: {lasso_zero if lasso_zero else "なし"}')


In [ ]:
# 係数比較
ols_plain = sm.OLS(y_train, sm.add_constant(X_train[FEAT_REDUCED])).fit()
ols_sc    = __import__('sklearn.linear_model', fromlist=['LinearRegression']).LinearRegression()
ols_sc.fit(X_tr_sc, y_train)

coef_df = pd.DataFrame({
    '変数':       FEAT_REDUCED,
    'OLS':        ols_sc.coef_,
    'Ridge':      ridge_cv.coef_,
    'Lasso':      lasso_cv.coef_,
    'ElasticNet': enet_cv.coef_,
})

fig, ax = plt.subplots(figsize=(12, 5))
x, w = np.arange(len(FEAT_REDUCED)), 0.2
for j, (col, color) in enumerate(zip(['OLS','Ridge','Lasso','ElasticNet'],
                                      ['#2980b9','#e67e22','#27ae60','#9b59b6'])):
    ax.bar(x + (j-1.5)*w, coef_df[col], w, label=col, color=color, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(FEAT_REDUCED, rotation=20, ha='right')
ax.set(ylabel='標準化係数', title='OLS / Ridge / Lasso / Elastic Net 係数比較')
ax.legend(); plt.tight_layout(); plt.show()


---
## 8. AIC / BIC によるモデル選択

$$\text{AIC} = 2k - 2\ln\hat{L}, \qquad \text{BIC} = k\ln n - 2\ln\hat{L}$$

Raschka ch09 で紹介されている変数セットから出発し、段階的に変数を追加・削除して比較する。


In [ ]:
model_specs = {
    'M0: 面積のみ':            ['Gr Liv Area'],
    'M1: Raschka基本':         ['Overall Qual','Overall Cond','Gr Liv Area',
                                 'Central Air','Total Bsmt SF'],
    'M2: M1+住設備':           ['Overall Qual','Overall Cond','Gr Liv Area',
                                 'Central Air','Total Bsmt SF',
                                 'Garage Area','Full Bath'],
    'M3: M2+立地年':           ['Overall Qual','Overall Cond','Gr Liv Area',
                                 'Central Air','Total Bsmt SF',
                                 'Garage Area','Full Bath','Year Built','Lot Area'],
    'M4: 削減後（VIF対処済）':  FEAT_REDUCED,
    'M5: M4+多重共線性変数戻': FEATURE_COLS,
}

results = []
for name, cols in model_specs.items():
    m = sm.OLS(y, sm.add_constant(df[cols])).fit()
    results.append({
        'モデル': name, 'k': len(cols)+1,
        'R²': round(m.rsquared, 4), 'Adj-R²': round(m.rsquared_adj, 4),
        'AIC': round(m.aic, 1), 'BIC': round(m.bic, 1),
        'RMSE': round(np.sqrt(m.mse_resid), 5),
    })

res_df = pd.DataFrame(results)
# 最良を強調
aic_best = res_df.loc[res_df['AIC'].idxmin(), 'モデル']
bic_best = res_df.loc[res_df['BIC'].idxmin(), 'モデル']
print(res_df.to_string(index=False))
print(f'\n✅ AIC 最良: {aic_best}')
print(f'✅ BIC 最良: {bic_best}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = plt.cm.Set2(np.linspace(0, 1, len(res_df)))
for ax, metric in zip(axes, ['AIC', 'BIC']):
    bars = ax.bar(res_df['モデル'], res_df[metric], color=bar_colors,
                  edgecolor='white', width=0.6)
    best = res_df[metric].idxmin()
    bars[best].set_edgecolor('gold'); bars[best].set_linewidth(3)
    ax.set_xticklabels(res_df['モデル'], rotation=20, ha='right', fontsize=8)
    ax.set(ylabel=metric, title=f'{metric}（金枠=最良）')
    for bar, val in zip(bars, res_df[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.0f}', ha='center', va='bottom', fontsize=7)
plt.suptitle('AIC / BIC モデル比較', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 9. 最終モデル評価：交差検証

AIC/BIC で選ばれた最良モデルを 10-fold CV で評価する。  
予測値を対数から元のスケール（ドル）に戻して RMSE を報告する。


In [ ]:
best_cols = model_specs[aic_best]
X_best    = df[best_cols].values
y_orig    = df[TARGET].values  # 元スケール

kf = KFold(n_splits=10, shuffle=True, random_state=42)

rmses_log  = []
rmses_orig = []
r2s        = []

for train_idx, val_idx in kf.split(X_best):
    Xtr, Xval = X_best[train_idx], X_best[val_idx]
    ytr, yval_log  = y.values[train_idx], y.values[val_idx]
    yval_orig = y_orig[val_idx]

    m = sm.OLS(ytr, sm.add_constant(Xtr)).fit()
    pred_log  = m.predict(sm.add_constant(Xval))
    pred_orig = np.exp(pred_log)

    rmses_log.append(np.sqrt(mean_squared_error(yval_log, pred_log)))
    rmses_orig.append(np.sqrt(mean_squared_error(yval_orig, pred_orig)))
    r2s.append(r2_score(yval_log, pred_log))

print(f'10-fold CV 結果（最良モデル: {aic_best}）')
print(f'  RMSE（log スケール）: {np.mean(rmses_log):.5f} ± {np.std(rmses_log):.5f}')
print(f'  RMSE（USD）         : ${np.mean(rmses_orig):,.0f} ± ${np.std(rmses_orig):,.0f}')
print(f'  R²                  : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}')


In [ ]:
# 最終モデルを全データで学習して予測値 vs 実測値を可視化
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    df[best_cols], y, test_size=0.2, random_state=42)
y_test_orig = np.exp(y_test_f.values)

final_m  = sm.OLS(y_train_f, sm.add_constant(X_train_f)).fit()
pred_log = final_m.predict(sm.add_constant(X_test_f))
pred_usd = np.exp(pred_log)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 実測 vs 予測（対数スケール）
axes[0].scatter(y_test_f, pred_log, alpha=0.3, s=15, color='#2980b9', edgecolors='white', lw=0.3)
lims = [y_test_f.min(), y_test_f.max()]
axes[0].plot(lims, lims, 'r--', lw=1.5)
axes[0].set(xlabel='実測 log(SalePrice)', ylabel='予測 log(SalePrice)',
            title=f'実測 vs 予測（log）\nR²={r2_score(y_test_f, pred_log):.4f}')

# 実測 vs 予測（USD）
axes[1].scatter(y_test_orig/1000, pred_usd/1000, alpha=0.3, s=15, color='#27ae60',
                edgecolors='white', lw=0.3)
lims_usd = [y_test_orig.min()/1000, y_test_orig.max()/1000]
axes[1].plot(lims_usd, lims_usd, 'r--', lw=1.5)
rmse_usd = np.sqrt(mean_squared_error(y_test_orig, pred_usd))
axes[1].set(xlabel='実測 (千ドル)', ylabel='予測 (千ドル)',
            title=f'実測 vs 予測（USD）\nRMSE=${rmse_usd:,.0f}')

plt.suptitle(f'最終モデル評価（{aic_best}）', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 10. まとめ

### statsmodels で得られた本格的な情報

| 分析 | 得られた知見 |
|------|------------|
| OLS サマリ | 各変数の係数・標準誤差・t値・p値・95%CI |
| VIF | `Gr Liv Area`-`TotRms AbvGrd`、`Total Bsmt SF`-`1st Flr SF`、`Garage Area`-`Garage Cars` で多重共線性 |
| RESET 検定 | 非線形関係の見落としを検出 |
| Breusch-Pagan 検定 | 不均一分散の検出 → ロバスト SE または WLS で対処 |
| Durbin-Watson | 残差の系列相関の確認 |
| Cook's D / Leverage | 外れ値・影響点の特定 |
| AIC / BIC | 変数追加の情報利得とペナルティのバランス評価 |

### 演習問題
1. `Year Built` の代わりに `Age = Yr Sold - Year Built`（築年数）を作り、係数解釈がどう変わるか比較せよ。
2. Lasso が選んだ変数のみで OLS を再推定し、フル OLS モデルの AIC と比較せよ（"Lasso + OLS" 戦略）。
3. 異常に高い Cook's D を持つ観測値を除外したモデルと除外しないモデルの係数を比較せよ。
4. `Neighborhood`（地域）変数をダミー変数化して投入し、AIC はどう変化するか調べよ。
5. 販売価格を log 変換したモデルと変換しないモデルの残差診断を比較し、変換の正当性を統計的に議論せよ。
